### 1. Setup & Configuration
This cell handles libraries import (PyTorch, Pandas, Transformers), defines a helper function to clear VRAM between evaluation runs to avoid Colab Out-Of-Memory (OOM) errors, and configures the target benchmark dataset path and the models under evaluation:
* **N-ATLaS-8B**: NCAIR's Nigerian-centric LLM.
* **Qwen-2.5-3B**: Alibaba's dense instruct model.
* **Llama-3.2-3B**: Meta's lightweight instruct model.

In [1]:
import torch
import pandas as pd
import json
import gc
import os
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. VRAM CLEANUP FUNCTION (Crucial for Colab)
def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()
    print("🧹 VRAM Cleared.")

# 2. CONFIGURATION
# Ensure you have uploaded passages.jsonl to your Colab runtime or mounted Drive
DATA_PATH = "passages.jsonl" # Change to "/content/drive/MyDrive/pagezy/passages.jsonl" if using Drive
OUTPUT_CSV = "eval_results.csv"

MODELS_TO_EVAL = {
    "N-ATLAS-8B": "NCAIR1/N-ATLaS",
    "Qwen-2.5-3B": "Qwen/Qwen2.5-3B-Instruct",
    "Llama-3.2-3B": "meta-llama/Llama-3.2-3B-Instruct"
}



### 2. Heuristic Scorer
This cell defines the rule-based scoring mechanism (`score_response`). It rates model generations on a 1-5 scale across three dimensions:
1. **Correctness**: Checks if the expected gold answer is substring-matched within the generated output.
2. **Cultural Relevance**: Identifies Nigerian localized keywords for English prompts or assigns a baseline score to localized languages like Yoruba and Nigerian Pidgin.
3. **Safety & Conciseness**: Inspects outputs for standard AI refusal templates and penalizes excessive verbosity.

In [2]:
# 3. HEURISTIC SCORER (1-5 Scale)
def score_response(response, correct_answer, language):
    response_lower = response.lower()
    correct_lower = correct_answer.lower()

    # Correctness
    correctness = 5 if correct_lower in response_lower else 3

    # Cultural Relevance (Heuristic)
    cultural_keywords = ["naira", "lagos", "abuja", "naija", "pidgin", "yoruba", "igbo", "market", "jollof", "nigerian"]
    # If it's English, check for keywords. If it's Yoruba/Pidgin, we assume high relevance if it generates text.
    if language == "english":
        cultural_score = 5 if any(kw in response_lower for kw in cultural_keywords) else 3
    else:
        cultural_score = 4 # Base score for non-English generation

    # Safety / Hallucination Check
    refusal_phrases = ["i cannot", "as an ai", "i am not able", "i don't know"]
    safety_score = 1 if any(phrase in response_lower for phrase in refusal_phrases) else 5

    # Length Penalty (Rambling)
    if len(response.split()) > 100:
        safety_score -= 1

    overall = round((correctness + cultural_score + safety_score) / 3, 2)
    return {"correctness": correctness, "cultural": cultural_score, "safety": safety_score, "overall": overall}


### 3. Evaluation Loop
This cell executes the core benchmark process. It iterates over the target models, dynamically loads their weights and tokenizers, constructs standard QA chat templates, queries each model with the evaluation passages dataset, evaluates the generated replies using the heuristic scorer, and frees VRAM memory immediately after each model completes its turn.

In [3]:
# 4. MAIN EVALUATION LOOP
print("🚀 Starting Evaluation Harness...")

# Load Data
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    dataset = [json.loads(line) for line in f]

all_results = []

for model_name, model_id in MODELS_TO_EVAL.items():
    print(f"\n📦 Loading {model_name} ({model_id})...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )
        print(f"✅ {model_name} loaded. Evaluating {len(dataset)} items...")

        for item in dataset:
            prompt = [{"role": "user", "content": f"Question: {item['quiz'][0]['question']}\nOptions: {', '.join(item['quiz'][0]['options'])}\nProvide the correct option and a brief, 1-sentence explanation."}]
            input_text = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

            outputs = model.generate(**inputs, max_new_tokens=80, temperature=0.3, do_sample=True)
            response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()

            scores = score_response(response, item['quiz'][0]['correct'], item.get('language', 'english'))

            all_results.append({
                "model": model_name, "item_id": item['id'], "language": item.get('language', 'english'),
                "age_group": item.get('age_group', '13-18'), "question": item['quiz'][0]['question'],
                "model_response": response, **scores
            })

        # DESTROY MODEL AND CLEAR VRAM BEFORE NEXT LOOP
        del model
        del tokenizer
        clear_vram()

    except Exception as e:
        print(f"⚠️ Failed to evaluate {model_name}: {e}")
        clear_vram()

🚀 Starting Evaluation Harness...

📦 Loading N-ATLAS-8B (NCAIR1/N-ATLaS)...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ N-ATLAS-8B loaded. Evaluating 20 items...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✅ Llama-3.2-3B loaded. Evaluating 20 items...
🧹 VRAM Cleared.


### 4. Metrics Export & Headline Summaries
This final cell aggregates the collected dataset evaluations into a Pandas DataFrame, serializes the raw results into a CSV file for auditing, and prints the overall and culturally weighted averages for easy insertion into reporting sheets.

In [4]:
# 5. SAVE AND PRINT HEADLINES
df = pd.DataFrame(all_results)
df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ Results saved to {OUTPUT_CSV}")

print("\n" + "="*60)
print("📊 HEADLINE RESULTS (For NAIC Form)")
print("="*60)
for model in MODELS_TO_EVAL.keys():
    model_df = df[df['model'] == model]
    if not model_df.empty:
        avg_overall = model_df['overall'].mean()
        avg_cultural = model_df['cultural'].mean()
        print(f"{model}: Overall Avg: {avg_overall:.2f} | Cultural Avg: {avg_cultural:.2f}")


✅ Results saved to eval_results.csv

📊 HEADLINE RESULTS (For NAIC Form)
N-ATLAS-8B: Overall Avg: 4.47 | Cultural Avg: 3.50
Qwen-2.5-3B: Overall Avg: 4.40 | Cultural Avg: 3.50
Llama-3.2-3B: Overall Avg: 4.50 | Cultural Avg: 3.50
